# 📘 Business KPI & Client Communication Handbook
### Based on the Global Superstore Sales & Profit Analysis Project

This notebook is a personal reference guide compiled from a real client engagement.
It captures **how to translate raw transactional data into business-level insights**
that a CEO or client actually cares about.

**Goal of this notebook:** Never dump 50 charts on a client. Find the numbers that
change what management does next.


---
## 1️⃣ Business Framing — Understand Before You Analyze

Before writing a single line of analysis code, answer these 5 questions.
This is what separates a "data cleaner" from a "business analyst."

| # | Question | Why it matters |
|---|---|---|
| 1 | What business is this? | Sets context — you can't judge "good" or "bad" numbers without knowing the business model |
| 2 | What problem might the company have? | Gives your analysis a *purpose*, not just exploration |
| 3 | Why is solving this problem important? | This is the "so what" — connects data to business risk/opportunity |
| 4 | What decisions might management need to make? | Analysis is worthless if it doesn't lead to a decision |
| 5 | How can data help? | Defines what to measure: diagnose → root cause → prevent |


In [ ]:
# STEP 1: Understand the business scope before analyzing anything
print("Categories:", df['Category'].unique())
print("Markets:", df['Market'].unique())
print("Segments:", df['Segment'].unique())
print("Total unique products:", df['Product Name'].nunique())

**Answers for Global Superstore (example):**

1. **What business is this?** A multi-national retail company selling Furniture,
   Technology, and Office Supplies across multiple regions (US, EU, APAC, LATAM,
   Africa, EMEA, Canada).

2. **What problem might the company have?** Sales are growing, but profitability
   is uneven — some categories/regions make money, others quietly lose it.

3. **Why is solving this important?** Growth without profit is a *growth trap* —
   scaling revenue while losses scale with it is not sustainable.

4. **What decisions might management need to make?** Discount policy, regional
   strategy (invest vs restructure), product portfolio decisions, resource allocation.

5. **How can data help?** Diagnose where the problem is → find the root cause →
   predict/prevent future losses.


---
## 2️⃣ Diagnosing the Problem — Proving It With Numbers

Once you have a hypothesis (e.g. "profitability is uneven"), prove it with data.


In [ ]:
# Overall profit margin — the single most important health number
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
overall_margin = (total_profit / total_sales) * 100
print(f"Overall Profit Margin: {overall_margin:.1f}%")

# Category-wise margin — shows WHERE the imbalance is
category_summary = df.groupby('Category').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum')
)
category_summary['Margin%'] = (category_summary['Total_Profit'] / category_summary['Total_Sales']) * 100
print(category_summary)

**Insight:** Furniture had the weakest margin (6.9%) vs Technology/Office Supplies (~14%).
This single table is more useful to a CEO than 10 charts — it directly points to *where* to look next.


---
## 3️⃣ Root Cause Analysis — Discount Policy Example

A "policy recommendation" isn't one line of code — it's a derived threshold from data.
Below is the technique used to find *at what discount level orders start losing money*.


In [ ]:
# Technique 1: Bin discounts into ranges and check average profit per range
bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 1.0]
labels = ['0-10%', '10-20%', '20-30%', '30-40%', '40-50%', '50%+']

df['Discount_Range'] = pd.cut(df['Discount'], bins=bins, labels=labels)

discount_analysis = df.groupby('Discount_Range')['Profit'].mean()
print(discount_analysis)

In [ ]:
# Technique 2: Visualize where the average profit crosses into negative territory
plt.figure(figsize=(8,5))
discount_analysis.plot(kind='bar', color='steelblue')
plt.axhline(0, color='red', linestyle='--')
plt.title('Average Profit by Discount Range')
plt.xlabel('Discount Range')
plt.ylabel('Average Profit')
plt.show()

In [ ]:
# Technique 3: Quantify the risk — % of orders that are unprofitable per discount range
loss_pct = df.groupby('Discount_Range').apply(lambda x: (x['Profit'] < 0).mean() * 100)
print(loss_pct)

**Client-ready recommendation (the story, not the code):**

> "Data shows that once discount crosses **30–40%**, orders almost always become
> unprofitable. We recommend a company-wide discount ceiling of ~30%, which protects
> margin while keeping sales volume largely intact."

**Root cause found in this project:** *Tables* sub-category had a 29.1% average
discount (vs 14% for everything else), causing 57.6% of its orders to be unprofitable —
making it the only sub-category losing money overall (-$64,083).


---
## 4️⃣ From "Thousands of Transactions" to a CEO Scorecard

**Client requirement:** *"We don't want a report with 50 charts. We only want the
metrics that truly tell us whether our business is healthy."*

**Rule of thumb:** 5–8 core metrics, grouped into 4 categories:
1. **Growth Health** — is the business growing?
2. **Profitability Health** — is it making money?
3. **Efficiency Health** — are operations sustainable?
4. **Risk / Red Flags** — where is the danger?


In [ ]:
# Core Business Health Metrics — all in one place
total_revenue = df['Sales'].sum()
total_profit = df['Profit'].sum()
total_orders = df['Order ID'].nunique()
profit_margin = (total_profit / total_revenue) * 100
avg_order_value = total_revenue / total_orders
avg_discount = df['Discount'].mean() * 100
unprofitable_pct = (df['Profit'] < 0).mean() * 100

print(f"Total Revenue: ${total_revenue:,.0f}")
print(f"Total Profit: ${total_profit:,.0f}")
print(f"Profit Margin: {profit_margin:.1f}%")
print(f"Total Orders: {total_orders:,}")
print(f"Average Order Value: ${avg_order_value:.2f}")
print(f"Average Discount: {avg_discount:.1f}%")
print(f"% Unprofitable Transactions: {unprofitable_pct:.1f}%")

In [ ]:
# Year-over-year growth (requires a date column)
yearly = df.groupby('Order Year').agg(
    Revenue=('Sales', 'sum'),
    Profit=('Profit', 'sum')
)
yearly['Revenue_Growth%'] = yearly['Revenue'].pct_change() * 100
yearly['Profit_Growth%'] = yearly['Profit'].pct_change() * 100
print(yearly)

In [ ]:
# The only chart type this kind of report usually needs — a trend line, not 50 charts
plt.figure(figsize=(8,5))
plt.plot(yearly.index, yearly['Revenue'], marker='o', label='Revenue')
plt.plot(yearly.index, yearly['Profit'], marker='o', label='Profit')
plt.title('Revenue & Profit Trend')
plt.legend()
plt.show()

---
## 5️⃣ 10 KPIs a CEO Should Monitor Every Month

| # | KPI | Why It Matters |
|---|---|---|
| 1 | Monthly Revenue | Basic pulse of the business — top-line growth |
| 2 | Monthly Profit & Profit Margin % | Growth without profit is unhealthy growth |
| 3 | Total Orders (Volume) | Demand trend, separate from revenue value |
| 4 | Average Order Value (AOV) | Pricing / upsell health |
| 5 | Average Discount % | Leading indicator of margin erosion |
| 6 | % of Unprofitable Orders | Early warning system for hidden losses |
| 7 | Revenue Growth Rate (MoM %) | Shows momentum, not just a snapshot |
| 8 | Top & Bottom Performing Category/Region | Where to invest, where to fix |
| 9 | Customer Segment Contribution | Reveals dependency/concentration risk |
| 10 | Shipping Cost as % of Revenue | Operational efficiency signal |


In [ ]:
# Example: monthly-level calculation pattern used for most KPIs above
monthly_revenue = df.groupby(df['Order Date'].dt.to_period('M'))['Sales'].sum()
monthly_profit = df.groupby(df['Order Date'].dt.to_period('M'))['Profit'].sum()
monthly_margin = (monthly_profit / monthly_revenue) * 100
monthly_orders = df.groupby(df['Order Date'].dt.to_period('M'))['Order ID'].nunique()
aov = monthly_revenue / monthly_orders
monthly_discount = df.groupby(df['Order Date'].dt.to_period('M'))['Discount'].mean() * 100
monthly_loss_pct = df.groupby(df['Order Date'].dt.to_period('M')).apply(lambda x: (x['Profit'] < 0).mean() * 100)
revenue_growth = monthly_revenue.pct_change() * 100

---
## 6️⃣ Narrowing Down: Top 5 KPIs

| KPI | Why This Made the Cut |
|---|---|
| 1. Revenue (+ Growth %) | The most basic pulse — without it, nothing else matters |
| 2. Profit Margin % | Prevents revenue growth from masking a loss-making business |
| 3. % of Unprofitable Orders | Early warning system — catches hidden problems before they scale |
| 4. Average Discount % | Leading indicator — moves *before* margin does |
| 5. Average Order Value (AOV) | Tells you if growth is from volume or value — different strategies follow |

**Why the other 5 were dropped:**
- *Total Orders* → already implied by Revenue + AOV
- *Top/Bottom Category* → a drill-down tool, not a monthly headline metric
- *Segment Contribution* → better suited for quarterly review
- *Shipping Cost %* → more relevant to Operations/COO than CEO
- *Revenue Growth Rate* → merged into KPI #1


---
## 7️⃣ The 60-Second CEO Briefing — Top 3 KPIs

**Scenario:** The CEO has only 60 seconds. What do you show first?

| Order | KPI | Why First |
|---|---|---|
| 1 | Revenue (+ Growth %) | Answers the CEO's instinctive first question: "Are we growing or not?" |
| 2 | Profit Margin % | Immediately clarifies if that growth is *real* or *hollow* |
| 3 | % of Unprofitable Orders | The risk radar — flags hidden problems even when margin looks fine |

**The compressed story these 3 numbers tell:**
> "How much are we growing (#1) → is that growth profitable (#2) → is there a hidden risk (#3)?"

**Why AOV and Discount % were left out of the 60-second version:**
They are "why" level detail, not "what" level status. In the first 60 seconds, a CEO
needs **triage**, not root cause. If #2 or #3 shows a red flag, *then* the CEO asks "why?"
— and that's when AOV/Discount enter the conversation.


---
## 8️⃣ The Core Definition to Remember

> ### "A KPI is valuable only if it **changes what management does next.**"

If a number is interesting, accurate, or nice to calculate — but looking at it doesn't
change any decision — it's not a KPI. It's just a data point.

**A simple test:**
> If I show this number to the CEO and their reaction is just *"ok"* — it's not a KPI.
> If their reaction is *"we need a meeting on this tomorrow"* — it's a real KPI.


---
## 9️⃣ The Owner Mindset Reframe

**Don't think:** *"What KPIs exist?"*
**Think:** *"If I owned this company, what would I check every Monday morning
before making decisions?"*

An owner doesn't scan 10 metrics every Monday. An owner asks **3 survival questions**:

1. **"How much money came in last week, and is the trend up or down?"**
   → Revenue this week vs last week

2. **"Am I actually making money from what's selling, or am I fooling myself?"**
   → Profit Margin this week

3. **"Is there something that looks fine on the surface but is quietly hurting me?"**
   → % Unprofitable Orders / worst-performing segment

**The real difference between "consultant thinking" and "owner thinking":**
- A consultant asks: *"Which metrics are important?"*
- An owner asks: *"What do I need to know right now to act with confidence — or
  act immediately if something's wrong?"*

An owner doesn't have time to explore metrics. They need **instant clarity**:
is everything fine, or not — and if not, exactly where to look.


---
## 🔑 Quick-Reference Summary

1. **Always frame the business first** (5 questions) before touching code.
2. **Prove problems with numbers**, then find the root cause (e.g. discount thresholds).
3. **Fewer, sharper metrics beat more charts.** 5–8 for a monthly scorecard, 3 for a
   60-second briefing.
4. **Every KPI must justify itself** by the test: *does it change a decision?*
5. **Think like an owner, not a metrics catalog** — ask what you'd need to know
   Monday morning to act with confidence.

---
*Compiled as a personal handbook from the Global Superstore Sales & Profit Analysis project
and subsequent client-communication practice exercises.*


🔑 Quick-Reference Summary
Always frame the business first (5 questions) before touching code.
Prove problems with numbers, then find the root cause (e.g. discount thresholds).
Fewer, sharper metrics beat more charts. 5–8 for a monthly scorecard, 3 for a 60-second briefing.
Every KPI must justify itself by the test: does it change a decision?
Think like an owner, not a metrics catalog — ask what you'd need to know Monday morning to act with confidence.

🔑 Quick-Reference Summary
Always frame the business first (5 questions) before touching code.
Prove problems with numbers, then find the root cause (e.g. discount thresholds).
Fewer, sharper metrics beat more charts. 5–8 for a monthly scorecard, 3 for a 60-second briefing.
Every KPI must justify itself by the test: does it change a decision?
Think like an owner, not a metrics catalog — ask what you'd need to know Monday morning to act with confidence.

🔑 Quick-Reference Summary
Always frame the business first (5 questions) before touching code.
Prove problems with numbers, then find the root cause (e.g. discount thresholds).
Fewer, sharper metrics beat more charts. 5–8 for a monthly scorecard, 3 for a 60-second briefing.
Every KPI must justify itself by the test: does it change a decision?
Think like an owner, not a metrics catalog — ask what you'd need to know Monday morning to act with confidence.

🔑 Quick-Reference Summary
Always frame the business first (5 questions) before touching code.
Prove problems with numbers, then find the root cause (e.g. discount thresholds).
Fewer, sharper metrics beat more charts. 5–8 for a monthly scorecard, 3 for a 60-second briefing.
Every KPI must justify itself by the test: does it change a decision?
Think like an owner, not a metrics catalog — ask what you'd need to know Monday morning to act with confidence.

🔑 Quick-Reference Summary
Always frame the business first (5 questions) before touching code.
Prove problems with numbers, then find the root cause (e.g. discount thresholds).
Fewer, sharper metrics beat more charts. 5–8 for a monthly scorecard, 3 for a 60-second briefing.
Every KPI must justify itself by the test: does it change a decision?
Think like an owner, not a metrics catalog — ask what you'd need to know Monday morning to act with confidence.

🔑 Quick-Reference Summary
Always frame the business first (5 questions) before touching code.
Prove problems with numbers, then find the root cause (e.g. discount thresholds).
Fewer, sharper metrics beat more charts. 5–8 for a monthly scorecard, 3 for a 60-second briefing.
Every KPI must justify itself by the test: does it change a decision?
Think like an owner, not a metrics catalog — ask what you'd need to know Monday morning to act with confidence.

🔑 Quick-Reference Summary
Always frame the business first (5 questions) before touching code.
Prove problems with numbers, then find the root cause (e.g. discount thresholds).
Fewer, sharper metrics beat more charts. 5–8 for a monthly scorecard, 3 for a 60-second briefing.
Every KPI must justify itself by the test: does it change a decision?
Think like an owner, not a metrics catalog — ask what you'd need to know Monday morning to act with confidence.